# Is it a planet or the star? Find an activity cycle in twenty years of data

[Open this notebook in Colab](https://colab.research.google.com/github/Rakiura-Inc/limits-of-habitability/blob/main/notebooks/02_planet_or_star.ipynb)

A planet search measures a star's velocity to within a metre per second or better. At that precision the star's own magnetic activity shows up too, and a slow activity cycle can look exactly like a slow planet. In this notebook you find an activity cycle in one star's twenty-year record, then decide what it would mean for a planet search on that star.

Run the cells in order (Shift+Enter). Boxes headed **Write down** ask you questions. Answer each one in the cell underneath by replacing its `...`, then run that cell: the notebook keeps your answer. At the end of each part, one cell puts all your answers into a single table and saves it with your plots. That table is what you hand in on the step.

In [ ]:
# Setup. Run this first. In Colab it installs what the notebook needs.
# If Colab asks you to restart the session afterwards, restart it and run the notebook from the top.
import importlib.util

# habtrain: a small package written for these notebooks. It downloads the data, fits orbits and draws the plots.
if importlib.util.find_spec("habtrain") is None:
    %pip install -q "habtrain @ git+https://github.com/Rakiura-Inc/limits-of-habitability@main"

# rakiura: the optional helper that connects this notebook to your step in Rakiura.
if importlib.util.find_spec("rakiura") is None:
    %pip install -q "rakiura @ git+https://github.com/Rakiura-Inc/rakiura-py@main"

import numpy as np                 # arrays and arithmetic
import matplotlib.pyplot as plt    # plots
import habtrain as ht              # the notebooks' package, used as ht.data, ht.fit, ht.plot and so on

# Which notebook this is, so your hand-in records exactly what you ran.
NOTEBOOK = "notebooks/02_planet_or_star.ipynb"
MODE = ht.portal.connect(NOTEBOOK)

> **Connect to Rakiura (optional)**
>
> Connect, and Rakiura ticks off each part as you finish it, with your plots on the step. You still attach your answers table and press Hand in on the step yourself. Skip it and everything still works: you attach the saved files on the step.
>
> To connect, run the cell below, open the link it prints, check the code matches the one shown, and click Allow. In Colab you do this once per session. To skip the click next time, add your key as a Colab secret named `RAKIURA_KEY` (the key icon on the left; your key is in Rakiura under Settings > Notebook key).

In [ ]:
# Connect to Rakiura (optional). Open the link it prints, check the code matches, click Allow.
MODE = ht.portal.connect(NOTEBOOK, sign_in=True)

## Why the star matters

A star's magnetic activity rises and falls over years, the way the Sun's does roughly every 11 years. A star's light, spread out by colour into a spectrum, has thin dark lines where atoms in the star absorb particular colours. Activity changes the shapes of some of those lines, and a velocity measured from them moves with it.

Activity is measured from two of those lines, made by calcium and named H and K (this K has nothing to do with the wobble K in notebook 1). Their centres brighten when a star is more active. The survey you use here reports it as an **S-value**: higher means more active. It measured 710 nearby stars with the HIRES spectrograph on Keck from 2005 to 2023, 52,372 measurements in all ([Isaacson et al. 2024](https://arxiv.org/abs/2406.17332)).

A **periodogram** tests every period in a range and gives each one a power: how well a repeating signal at that period fits the data. A tall, narrow peak is a signal that repeats.

In [ ]:
# Load every activity measurement in the survey: one row per spectrum, with the star's name, time and S-value.
series = ht.data.load_activity_series()
print(len(series), "measurements of", series.Name.nunique(), "stars")

## Worked example: 55 Cancri

HD 75732 is 55 Cancri. One of its planets, 55 Cnc f, is in the habitable zone catalog from notebook 1, where it is listed among the most promising habitable zone planets that do not transit. Here is its star's activity, one point per night, and the periodogram from 2 to 25 years.

In [ ]:
# 55 Cancri's measurements, averaged to one point per night (several spectra on one night count once).
night_55 = ht.data.nightly(series, "HD 75732")
fig = ht.plot.activity(night_55, "HD 75732 (55 Cancri)")
plt.show()

# The periodogram: how well a repeating signal fits, for every period from 2 to 25 years.
periods_55, power_55 = ht.fit.periodogram(night_55.year, night_55.S, 2, 25)

# The period with the most power is the tallest peak.
best_55 = periods_55[np.argmax(power_55)]
CREDIT = "Data: Isaacson et al. 2024, California Legacy Survey V, VizieR J/ApJS/274/35."
fig = ht.plot.periodogram(periods_55, power_55, "years", "HD 75732: which periods repeat?", mark=best_55, credit=CREDIT)
plt.show()

# How likely is a peak this tall from noise alone? Small is good.
print(f"Tallest peak: {best_55:.2f} years. Chance that noise alone makes a peak this tall: "
      f"{ht.fit.false_alarm(night_55.year, night_55.S, 2, 25):.1g}")

The tallest peak sits near 11 years: 55 Cancri has an activity cycle about as long as the Sun's.

How to judge a peak:
- **The longer the period, the less certain it is.** The record is about 19 years long, so a 16-year cycle is seen only about once, and a peak at the 25-year edge of the search is more likely a slow drift than a cycle. Look at the activity plot: can you see the rise and fall repeat?
- **It should stand clearly above the rest** of the periodogram, not just be the tallest of several similar bumps.
- **The chance from noise is a weak guide here.** It assumes each night is independent of the next, and a star's slow changes are not, so it comes out far too small: most stars score a tiny number, including many the survey found no cycle in. Use it to compare stars, not as a yes or no.

## Your turn: make a guess, then test it on a star

The survey searched for cycles only in stars it had measured at least 45 times since 2005. That is 285 stars. Some of them have a cycle and some do not.

The map below places each one by things you can know before you look for a cycle:
- **Across: colour** (B-V). Smaller is bluer and hotter, bigger is redder and cooler. The Sun is about 0.65.
- **Up: average activity** (log R'HK: the S-value scaled so stars of different colours can be compared). Higher is more active.
- **Point colour: how much its activity varies** over the years, as a percentage of its typical value.

55 Cancri, the worked example, is circled.

In [ ]:
# The 285 stars observed at least 45 times since 2005: the survey's own cut for searching for cycles.
stars = ht.data.well_sampled_stars(series)

# One line per star: its colour, average activity and how much the activity varies.
summary = ht.data.load_activity_summary()
summary = summary[summary.Name.isin(stars)]

# The star map, with 55 Cancri circled.
fig = ht.plot.star_map(summary, mark={"HD 75732": "55 Cancri"})
plt.show()

In [ ]:
# Optional: the same map, interactive. Hover over a point to see the star's name and details.
fig = ht.plot.star_map_interactive(summary)
if fig is not None:
    fig.show()

**Guess first**

Which kind of star do you expect to show a clear, repeating activity cycle: a quiet one, a middling one or a very active one? A quiet star barely changes; a very active one changes a lot, but not necessarily in a regular way. Write your guess and the reason in a few words, or "unsure". One star cannot settle it; the answer key shows what the survey found across all 285.

In [ ]:
# Your guess, before you choose a star: which kind of star shows a clear cycle, and why?
MY_EXPECTATION = ...   # for example "middling, because ...", or "unsure"
ht.record.predict("which_stars", MY_EXPECTATION)

Now choose a star that tests your guess. Move the two sliders to a spot on the map, and the five stars closest to it are circled and listed in the menu. Choose one from the menu. If you would rather not choose, the first option picks one at random.

In [ ]:
# Two sliders move a crosshair over the star map; the five closest stars appear in the menu below them.
chooser = ht.choose.star(summary, ht.plot.star_map, ht.data.nearest_stars)

> **Write down: the star that tests your guess**
>
> - Choose your star with the sliders and the menu above. Any star except 55 Cancri.
> - If the sliders do not appear: pick a point on the map and put its colour (across) in `MY_COLOUR` and its activity (up) in `MY_ACTIVITY`, run the cell to see the five closest stars, numbered, then put one of those numbers in `MY_PICK` and run it again.
>
> **Where to look:** the map and the menu above.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
MY_COLOUR = ...     # only if the sliders did not appear: the colour of your point on the map, for example 0.65
MY_ACTIVITY = ...   # and its activity, for example -4.8
MY_PICK = ...       # then a number from the five closest stars listed below

# The five closest stars to your point on the map, numbered (only needed if the sliders did not appear).
fallback = ht.data.nearest_stars(summary, ht.record.fill(MY_COLOUR, 0.65, "MY_COLOUR"), ht.record.fill(MY_ACTIVITY, -4.8, "MY_ACTIVITY"))     if chooser.value is None else ht.data.nearest_stars(summary, 0.65, -4.8)
if chooser.value is None:
    ht.record.numbered(fallback, ["Name", "BV", "logRhk", "variation", "Nobs"])

# Your star: what you chose in the menu, else the number you typed, else a random one.
star = ht.choose.resolve(chooser, MY_PICK, fallback, "Name", "star")

# Keep your guess and your star for the hand-in table.
ht.record.save("expectation", MY_EXPECTATION, "your guess: which kind of star shows a clear cycle, and why")
ht.record.save("my_star", star, "the star you picked to test it")

# Your star's activity, one point per night.
night = ht.data.nightly(series, star)
fig_activity = ht.plot.activity(night, star)
plt.show()
print(len(night), "nights of data")

**Predict first**

Look at the plot. Do you see a slow rise and fall that repeats? If so, roughly how many years from one peak to the next?

In [ ]:
# Your prediction from the plot alone: how many years from one peak to the next?
MY_GUESS = ...   # a number of years, or "no cycle", or "unsure"
ht.record.predict("cycle", MY_GUESS)

> **Write down: the period your star repeats at**
>
> - Work out your star's periodogram: put the code in `periods, power`.
> - What is the period of its tallest peak, in years? Put the code in `best`.
> - Does your star have an activity cycle? `"yes"`, `"no"` or `"unsure"`. Put it in `HAS_CYCLE`.
> - How sure are you, and why? `"sure"`, `"fairly sure"` or `"unsure"`, and a few words. Put it in `HOW_SURE`.
>
> **Where to look:** the periodogram your code draws, and the chance-from-noise number printed under it.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

<details><summary>Stuck? Open hint 1</summary>

Copy the worked example: `ht.fit.periodogram(night.year, night.S, 2, 25)` gives the periods and their power.

</details>

<details><summary>Stuck? Open hint 2</summary>

`periods[np.argmax(power)]` is the period with the most power.

</details>

In [ ]:
# This only runs once you have made your prediction above.
if ht.record.revealed("cycle"):
    periods, power = ...   # your code: the periodogram of your star, 2 to 25 years
    best = ...             # your code: the period of the tallest peak
    # Draw your periodogram and say how likely its tallest peak is from noise alone.
    if ht.record.filled(best):
        fig_periodogram = ht.plot.periodogram(periods, power, "years", f"{star}: which periods repeat?", mark=best, credit=CREDIT)
        plt.show()
        print(f"Tallest peak: {best:.2f} years. Chance that noise alone makes a peak this tall: "
              f"{ht.fit.false_alarm(night.year, night.S, 2, 25):.1g}")

HAS_CYCLE = ...   # "yes", "no" or "unsure"
HOW_SURE = ...    # "sure", "fairly sure" or "unsure", and why

# Keep the answers for your hand-in table.
ht.record.save("period", best if "best" in dir() else ..., "the period of the tallest peak (years)")
ht.record.save("has_cycle", HAS_CYCLE, "whether your star has an activity cycle")
ht.record.save("how_sure", HOW_SURE, "how sure you are, and why")

## What it would mean for a planet

Suppose a planet search on your star found a velocity signal with a period close to the one you just found.

**What you need**

Two checks do most of the work:
- **Does the activity rise and fall with the velocity?** The S-values come from the same spectra as the velocities, at the same times. If both show the same period, the star is the likelier cause.
- **Does the signal depend on which lines you measure?** A planet shifts every line in the spectrum by the same amount, because the whole star moves. Activity changes the shapes of particular lines, so its signal changes with the lines you use.

A third, cheaper one: a signal at the star's rotation period, or at a half or a third of it, is usually starspots coming back into view. The radial velocity chapter by Luque and Standing goes further.

> **Write down: whether you would believe the planet**
>
> - Would you call a velocity signal at your period a planet? Which check would you run first, and what result would make you drop the planet? One or two lines. Put it in `WOULD_YOU`.
>
> **Where to look:** your periodogram, and the checks above.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
WOULD_YOU = ...   # one or two lines, in quotes

# Keep the answer for your hand-in table.
ht.record.save("would_you", WOULD_YOU, "whether you would call a signal at that period a planet, and how you would check")

## Hand in

Run the cell below. It shows all your answers in one table. Check them, then hand in this step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `cycle-found`, and download them.

Once you have handed in, the step shows the answer key: the published result for your star, and which kind of star the survey found most often has a cycle, so you can see how your guess did.

In [ ]:
# Your plots and answers, saved as your hand-in.
figs = {"activity": fig_activity}
if "fig_periodogram" in dir():
    figs["periodogram"] = fig_periodogram
ht.record.handin("cycle-found", ["expectation", "my_star", "period", "has_cycle", "how_sure", "would_you"], figures=figs)